In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

In [ ]:
R_HOME = "/usr/lib/R"
R_USER = "/home/work/.local/lib/python3.10/site-packages/rpy2"

os.environ['R_HOME'] = R_HOME
os.environ['R_USER'] = R_USER

In [ ]:
DATA_PATH = "data/her2st"
PATHWAY_CSV = "data/KEGG/HSA_KEGG_Pathway.csv"
DATASET = "Her2st"
SAMPLE_NAME = "A1"
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
from train import train, seed_torch
from argparse import Namespace
from utils import build_her2st_data, calculate_adj_matrix, refine
from metrics import eval_mclust_ari
from sklearn.metrics import adjusted_rand_score
import cv2

In [ ]:
args = Namespace(
    dataset=DATASET,
    path=DATA_PATH,
    img_size=112,
    num_workers=4,
    last_dim=32,
    hidden_dim=128,
    lr=0.005,
    p_drop=0.3,
    result_file_name="her2st_experiment",
    w_g2g=0.1,
    w_i2i=0.1,
    w_recon=0,
    prob_edge_perturb=0.5,
    pct_edge_perturb=0.1,
    prob_node_drop=0.5,
    pct_node_drop=0.1,
    batch_size=32,
    epochs=100,
    device="cuda:0",
    log_name="her2st_log",
    name=SAMPLE_NAME,
    weight_decay=1e-4,
    checkpoint_path=None,
    temperature=1,
    csvfile = "data/KEGG/HSA_KEGG_Pathway.csv"
)

In [ ]:
train(args, args.name)

In [ ]:
adata, patches = build_her2st_data(os.path.join(DATA_PATH), SAMPLE_NAME)
print(f"Loaded Her2st data: {len(adata)} spots")

In [ ]:
ground_truth_labels = adata.obs['label'].values
valid_indices = ground_truth_labels != -1
ground_truth_labels = ground_truth_labels[valid_indices]
spatial_coords = adata.obsm['spatial'][valid_indices]
print(f"Valid spots with labels: {len(ground_truth_labels)}")
print(f"Number of clusters: {len(np.unique(ground_truth_labels))}")

In [ ]:
pred_label_path = os.path.join("output", f"{SAMPLE_NAME}_pred.csv")
pred_df = pd.read_csv(pred_label_path)
predicted_labels = pred_df['cluster_labels'].values
print(f"Loaded predictions: {len(predicted_labels)} spots")

In [ ]:
adj_2d = calculate_adj_matrix(
    x=spatial_coords[:, 0].tolist(), 
    y=spatial_coords[:, 1].tolist(),
    histology=False
)
num_nbs = 24 if len(spatial_coords) > 1000 else 4
refined_predicted_labels = refine(
    sample_id=adata.obs.index[valid_indices].tolist(), 
    pred=predicted_labels, 
    dis=adj_2d, 
    num_nbs=num_nbs
)
ari_score = adjusted_rand_score(ground_truth_labels, refined_predicted_labels)
print(f"Refined ARI Score: {ari_score:.4f}")

In [ ]:
contingency_matrix = confusion_matrix(ground_truth_labels, refined_predicted_labels)
cost_matrix = contingency_matrix.max() - contingency_matrix
row_ind, col_ind = linear_sum_assignment(cost_matrix)
mapping = {pred_idx: gt_idx for pred_idx, gt_idx in zip(col_ind, row_ind)}
print("Hungarian mapping completed.")

In [ ]:
df = pd.DataFrame({
    'spatial_x': spatial_coords[:, 0],
    'spatial_y': spatial_coords[:, 1],
    'Ground Truth': ground_truth_labels,
    'Prediction (Refined)': refined_predicted_labels
})
df['Prediction (Mapped)'] = df['Prediction (Refined)'].map(mapping).fillna(-1).astype(int)
canonical_order = sorted(np.unique(ground_truth_labels))
df['Ground Truth'] = pd.Categorical(df['Ground Truth'], categories=canonical_order, ordered=True)
df['Prediction (Mapped)'] = pd.Categorical(df['Prediction (Mapped)'], categories=canonical_order, ordered=True)
print("DataFrame prepared for visualization.")

In [ ]:
img_dir = os.path.join(DATA_PATH, 'data', 'ST-imgs', SAMPLE_NAME[0], SAMPLE_NAME)
img_files = os.listdir(img_dir)
histology_image_path = os.path.join(img_dir, img_files[0])
try:
    histology_image = mpimg.imread(histology_image_path)
    print(f"Histology image loaded: {histology_image_path}")
    print(f"Image shape: {histology_image.shape}")
except FileNotFoundError:
    print(f"Warning: Histology image not found at {histology_image_path}")
    histology_image = None

In [ ]:
plt.style.use('default')
fig, axes = plt.subplots(1, 2, figsize=(24, 10))
palette = sns.color_palette("bright", n_colors=len(canonical_order))
SPOT_SIZE = 100

ax1 = axes[0]
ax1.set_title('Spatial Plot - Ground Truth (Overlay)', fontsize=18, pad=20)
if histology_image is not None:
    ax1.imshow(histology_image, cmap='gray')
sns.scatterplot(data=df, x='spatial_x', y='spatial_y', hue='Ground Truth', 
                palette=palette, s=SPOT_SIZE, ax=ax1, legend=False, alpha=0.8, linewidth=0)
ax1.set_xlabel('')
ax1.set_ylabel('')
ax1.tick_params(axis='both', which='both', bottom=False, top=False, 
                left=False, right=False, labelbottom=False, labelleft=False)

ax2 = axes[1]
ax2.set_title('Spatial Plot - Mapped Prediction (Overlay)', fontsize=18, pad=20)
if histology_image is not None:
    ax2.imshow(histology_image, cmap='gray')
sns.scatterplot(data=df, x='spatial_x', y='spatial_y', hue='Prediction (Mapped)', 
                palette=palette, s=SPOT_SIZE, ax=ax2, legend=False, alpha=0.8, linewidth=0)
ax2.set_xlabel('')
ax2.set_ylabel('')
ax2.tick_params(axis='both', which='both', bottom=False, top=False, 
                left=False, right=False, labelbottom=False, labelleft=False)

ari_text = f"PathCLAST : ARI {ari_score:.4f}"
ax2.text(0.02, 0.98, ari_text, 
         transform=ax2.transAxes,
         fontsize=16, fontweight='bold', color='white', verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.6))

plt.tight_layout(pad=3.0)

output_filename = f"{SAMPLE_NAME}_temp{args.temperature}_visualization.png"
output_path = os.path.join(OUTPUT_DIR, output_filename)
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\nVisualization saved: {output_path}")

plt.show()